# FarmerVision — User-Centred Evaluation
### Notebook 18 · Milestone 5 §6.9

**Purpose:** Analyse expert review scores and usability session observations to report on answer quality from a farmer-facing perspective.

**Inputs:**
- `docs/internal/user_eval/expert_review_results.csv` — 20 answers, scored on 5 dimensions by two reviewer types
- `docs/internal/user_eval/usability_session_log.md` — 5 simulated usability sessions
- Automated metric scores from §6.4 (chrF++, RAGAS faithfulness) for the same 20 questions, cross-joined for correlation analysis

**Outputs:** §6.9 summary statistics, three figures (dimension bar chart, act_on breakdown, correlation heatmap)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from pathlib import Path

# ── Load expert review results ─────────────────────────────────────────────────
csv_path = Path("../docs/internal/user_eval/expert_review_results.csv")
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} reviewed answers")
df.head(3)

---
## 1. Dimension-level scores

In [ ]:
# ── Compute mean scores per dimension and reviewer type ───────────────────────
dims = ["D1", "D2", "D3", "D4", "D5"]
dim_labels = {
    "D1": "Factual Accuracy",
    "D2": "Completeness",
    "D3": "Clarity",
    "D4": "Actionability",
    "D5": "Language Appropriateness",
}

# Replace N/A strings with NaN for numerics
for col in [f"{d}_expert" for d in dims] + [f"{d}_farmer" for d in dims]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

summary = pd.DataFrame({
    "Dimension": [dim_labels[d] for d in dims],
    "Expert mean": [df[f"{d}_expert"].mean() for d in dims],
    "Farmer mean": [df[f"{d}_farmer"].mean() for d in dims],
    "Overall mean": [
        pd.concat([df[f"{d}_expert"], df[f"{d}_farmer"]]).mean()
        for d in dims
    ],
}).set_index("Dimension")

print(summary.round(2).to_string())

In [ ]:
# ── Fig 1: Mean scores per dimension (grouped bars) ───────────────────────────
x = np.arange(len(dims))
width = 0.3

fig, ax = plt.subplots(figsize=(9, 4.5))
b1 = ax.bar(x - width/2, summary["Expert mean"],  width, label="Domain expert",
            color="#2980b9", edgecolor="white")
b2 = ax.bar(x + width/2, summary["Farmer mean"],   width, label="Naïve farmer",
            color="#27ae60", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels([dim_labels[d] for d in dims], rotation=15, ha="right", fontsize=9)
ax.set_ylim(0, 5.5)
ax.set_ylabel("Mean score (1–5)")
ax.set_title("FarmerVision — Expert Review Scores by Dimension (n = 20 answers)", fontsize=11)
ax.axhline(3, color="grey", linestyle="--", linewidth=0.8, alpha=0.5)
ax.legend()

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.savefig("user_eval_dimension_scores.png", bbox_inches="tight")
plt.show()
print("Saved: user_eval_dimension_scores.png")

---
## 2. Binary act_on rating and inter-rater agreement

In [ ]:
# ── Encode act_on as binary ────────────────────────────────────────────────────
df["act_on_expert_bin"] = (df["act_on_expert"].str.strip().str.upper() == "YES").astype(int)
df["act_on_farmer_bin"] = (df["act_on_farmer"].str.strip().str.upper() == "YES").astype(int)

agreement_count = (df["act_on_expert_bin"] == df["act_on_farmer_bin"]).sum()
print(f"Expert YES: {df['act_on_expert_bin'].sum()}/20  "
      f"Farmer YES: {df['act_on_farmer_bin'].sum()}/20  "
      f"Agreement: {agreement_count}/20")

# Cohen's kappa
from sklearn.metrics import cohen_kappa_score
kappa = cohen_kappa_score(df["act_on_expert_bin"], df["act_on_farmer_bin"])
print(f"Cohen's kappa (act_on): {kappa:.3f}")
if kappa < 0.60:
    print("  ⚠  Kappa < 0.60 — tiebreak required for disagreement cases")
else:
    print("  ✓  Kappa ≥ 0.60 — acceptable inter-rater agreement")

In [ ]:
# ── Fig 2: act_on breakdown by language and intent ─────────────────────────────
act_by_lang = df.groupby("language")["act_on_expert_bin"].agg(
    yes="sum", total="count"
).assign(pct=lambda x: x["yes"] / x["total"])
act_by_intent = df.groupby("intent")["act_on_expert_bin"].agg(
    yes="sum", total="count"
).assign(pct=lambda x: x["yes"] / x["total"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(act_by_lang.index, act_by_lang["pct"], color="#2980b9", edgecolor="white")
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("Expert 'would act on this' rate")
axes[0].set_title("By language")
for i, (idx, row_) in enumerate(act_by_lang.iterrows()):
    axes[0].text(i, row_["pct"] + 0.02, f"{row_['pct']:.0%}\n(n={row_['total']})",
                 ha="center", fontsize=9)

axes[1].barh(act_by_intent.index, act_by_intent["pct"], color="#27ae60", edgecolor="white")
axes[1].set_xlim(0, 1.1)
axes[1].set_xlabel("Expert 'would act on this' rate")
axes[1].set_title("By intent class")
for i, (idx, row_) in enumerate(act_by_intent.iterrows()):
    axes[1].text(row_["pct"] + 0.02, i, f"{row_['pct']:.0%} (n={row_['total']})",
                 va="center", fontsize=8)

plt.suptitle("FarmerVision — 'Would act on this?' by language and intent (n = 20)", fontsize=10)
plt.tight_layout()
plt.savefig("user_eval_act_on_breakdown.png", bbox_inches="tight")
plt.show()
print("Saved: user_eval_act_on_breakdown.png")

---
## 3. Correlation with automated metrics

Cross-join the 20 reviewed answers with the automated metric scores (chrF++, RAGAS faithfulness) recorded in the Milestone 5 evaluation runs (§6.4). These are loaded from the per-row results file produced by `15_distill_model_evals.ipynb`.

**If the per-row file is not available**, the cell will print a warning and skip the correlation plot.

In [ ]:
# ── Load automated metric scores (from 15_distill_model_evals output) ─────────
auto_metric_path = Path("../outputs/distill_eval_per_row.csv")

if not auto_metric_path.exists():
    print(f"WARNING: {auto_metric_path} not found. Skipping correlation analysis.")
    print("To enable, export per-row results from notebook 15 with columns:")
    print("  question_id, chrf_pp, ragas_faithfulness, numeric_recall")
    auto_metrics = None
else:
    auto_metrics = pd.read_csv(auto_metric_path)
    # Merge on question_id
    merged = df.merge(auto_metrics, left_on="question_id", right_on="question_id", how="inner")
    print(f"Merged {len(merged)} rows with automated metrics")
    merged[["question_id", "D1_expert", "D4_expert", "act_on_expert_bin",
            "chrf_pp", "ragas_faithfulness", "numeric_recall"]].head()

In [ ]:
# ── Fig 3: Correlation heatmap ─────────────────────────────────────────────────
if auto_metrics is not None and 'merged' in dir():
    corr_cols = [
        "D1_expert", "D2_expert", "D3_expert", "D4_expert", "D5_expert",
        "act_on_expert_bin", "chrf_pp", "ragas_faithfulness", "numeric_recall"
    ]
    corr_cols_present = [c for c in corr_cols if c in merged.columns]
    corr = merged[corr_cols_present].corr()

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax, label="Pearson r")
    ticks = range(len(corr_cols_present))
    labels = [
        c.replace("_expert", "\n(exp)").replace("act_on_expert_bin", "act_on\n(exp)")
         .replace("chrf_pp", "chrF++").replace("ragas_faithfulness", "RAGAS faith.")
         .replace("numeric_recall", "num. recall")
        for c in corr_cols_present
    ]
    ax.set_xticks(ticks); ax.set_yticks(ticks)
    ax.set_xticklabels(labels, fontsize=8, rotation=45, ha="right")
    ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(corr_cols_present)):
        for j in range(len(corr_cols_present)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title("Expert scores vs. automated metrics — Pearson correlations", fontsize=10)
    plt.tight_layout()
    plt.savefig("user_eval_correlation_heatmap.png", bbox_inches="tight")
    plt.show()
    print("Saved: user_eval_correlation_heatmap.png")
else:
    print("Skipping correlation plot (automated metrics file not found).")

---
## 4. Usability session summary

In [ ]:
# Usability session data (transcribed from the log file)
sessions = pd.DataFrame([
    {"session": "S1", "pathway": "A", "language": "English",  "guardrail_correct": True,
     "pipeline_complete": True, "lang_match": "Yes", "numeric_grounded": True,
     "latency_s": 16.3, "farmer_trust": "Yes"},
    {"session": "S2", "pathway": "A", "language": "Hinglish", "guardrail_correct": True,
     "pipeline_complete": True, "lang_match": "Yes", "numeric_grounded": True,
     "latency_s": 14.8, "farmer_trust": "Yes"},
    {"session": "S3", "pathway": "B", "language": "Hindi",    "guardrail_correct": True,
     "pipeline_complete": True, "lang_match": "Yes", "numeric_grounded": True,
     "latency_s": 18.9, "farmer_trust": "Partial"},
    {"session": "S4", "pathway": "A", "language": "Hindi",    "guardrail_correct": True,
     "pipeline_complete": True, "lang_match": "Yes", "numeric_grounded": True,
     "latency_s": 15.6, "farmer_trust": "Yes"},
    {"session": "S5", "pathway": "C", "language": "N/A",      "guardrail_correct": True,
     "pipeline_complete": True, "lang_match": "N/A", "numeric_grounded": True,
     "latency_s": 0.03, "farmer_trust": "Partial"},
])

print("Usability session summary:")
print(sessions.to_string(index=False))

print(f"\nE2E completion: {sessions['pipeline_complete'].all()}")
print(f"Guardrail correct: {sessions['guardrail_correct'].all()}")
print(f"Farmer trust (Yes or Partial): {(sessions['farmer_trust'].isin(['Yes', 'Partial'])).all()}")
print(f"Mean latency (Pathway A+B): {sessions.loc[sessions['pathway'].isin(['A','B']), 'latency_s'].mean():.1f}s")

---
## 5. Report summary for §6.9

In [ ]:
expert_act_on_pct  = df["act_on_expert_bin"].mean()
farmer_act_on_pct  = df["act_on_farmer_bin"].mean()
best_dim  = summary["Overall mean"].idxmax()
worst_dim = summary["Overall mean"].idxmin()

print(f"""\n### §6.9 User-Centred Evaluation (Report Summary)

**Expert review (n = 20 answers, 2 reviewers each):**

| Dimension | Expert mean | Farmer mean |
|---|---|---|
{''.join(f'| {dim_labels[d]} | {summary.loc[dim_labels[d], "Expert mean"]:.2f} | {summary.loc[dim_labels[d], "Farmer mean"]:.2f} |\n' for d in dims)}

"Would you act on this answer?" — Expert: {expert_act_on_pct:.0%}   Farmer: {farmer_act_on_pct:.0%}  
Cohen's kappa on act_on: {kappa:.3f}

Highest-rated dimension: {best_dim} ({summary.loc[best_dim, 'Overall mean']:.2f}/5)  
Lowest-rated dimension:  {worst_dim} ({summary.loc[worst_dim, 'Overall mean']:.2f}/5)

**Usability sessions (5 scripted scenarios):**
- E2E completion: 5/5 (100%)
- Guardrail correct decisions: 5/5 (including 1 correct block)
- Language match: 4/4 sessions with text/image output
- Farmer trust "Yes" or "Partial": 5/5 — no session produced an untrustworthy output
- Both "Partial" trust ratings trace to interface gaps (confidence phrasing; raw yield number without context), not model errors
""")